# A rank-4 representation over the pasture $P$

This notebook verifies the pasture example in the paper "On the multiple exchange
property of matroids over hyperfields" by Nathan Bowler, Changxin Ding, and
Chayim Lowen. It checks that the support forms a matroid and that all three-term
Plücker relations hold, using the definition of a pasture representation in
Baker–Lorscheid. It does not check any multiple-exchange property or any
higher-term Plücker relation.

This notebook and its verification code were developed by Changxin Ding with
assistance from ChatGPT.

Choose **Run → Run All Cells**. All arithmetic is exact. The reusable functions
are in the adjacent file [p7_checks.py](p7_checks.py). The triangular-hyperfield
example is in a separate notebook and uses a separate checker.


## 1. The pasture and its encoding

The group $P^\times=\langle x\mid x^7=1\rangle$ has seven elements, and $-1=1$.
For exponents understood modulo $7$,
$$x^m+x^n+1\in N_P\quad\Longleftrightarrow\quad
(m,n)\notin\{(3,3),(0,4),(4,0)\}.$$
For three nonzero terms, multiply by $x^{-c}$ to obtain
$$x^a+x^b+x^c\in N_P\quad\Longleftrightarrow\quad
(a-c,b-c)\bmod 7\notin\{(3,3),(0,4),(4,0)\}.$$
The zero cases must also be tested: the pasture axioms and $-1=1$ imply
$0+a+b\in N_P$ if and only if $a=b$. Thus three zeros are null, one nonzero
term is not null, and two nonzero terms are null exactly when they are equal.

**Python encoding:** `P_ZERO` is `None`, `P_ONE` is the integer `0`, and
`x_power(k)` represents $x^k$ by its exponent modulo $7$.
The integer `0` therefore represents $1\in P$, **not** the zero of $P$.
Use the named constants in coordinate definitions. `p_label` displays the
mathematical values rather than their encodings.

This follows the pasture axioms in Baker–Lorscheid, *Foundations of matroids,
Part 1*, [arXiv:2008.00014v1](https://arxiv.org/abs/2008.00014v1),
Definition 2.1 and the introduction on p. 4.


In [1]:
from collections import Counter
from itertools import combinations
from p7_checks import (P_ZERO, P_ONE, x_power, p_label,
                       p_multiply, p_is_null, check_support_matroid,
                       check_three_term_plucker, check_p_matroid)


## 2. The example

Let $E=\{A,B,C,D,X_1,X_2,Y_1,Y_2\}$ and $r=4$. Put
$$
\begin{aligned}
\mathcal Z={}&\{\{A,B,X_1,X_2\},\{C,D,X_1,X_2\},
                 \{A,C,Y_1,Y_2\},\{B,D,Y_1,Y_2\}\},\\
\mathcal U={}&\{\{A,C,X_1,X_2\},\{A,D,X_1,X_2\},
                 \{B,C,X_1,X_2\},\{B,D,X_1,X_2\}\},\\
\mathcal V={}&\{\{A,B,Y_1,Y_2\},\{A,D,Y_1,Y_2\},
                 \{B,C,Y_1,Y_2\},\{C,D,Y_1,Y_2\}\}.
\end{aligned}
$$
For a four-subset $S$, let $S_0=S\cap\{A,B,C,D\}$ and say $S$ is mixed when
$|S\cap\{X_1,X_2\}|=|S\cap\{Y_1,Y_2\}|=1$. Define
$$
\chi(S)=\begin{cases}
0 & S\in\mathcal Z,\\
x & S\in\mathcal U,\\
x^2 & S\in\mathcal V,\\
x^6 & S\text{ mixed and }S_0\in\{\{A,B\},\{C,D\}\},\\
x^5 & S\text{ mixed and }S_0\in\{\{A,C\},\{B,D\}\},\\
1 & \text{otherwise}.
\end{cases}
$$
The first five cases are mutually exclusive. The code uses the labels `X1`,
`X2`, `Y1`, `Y2` for $X_1,X_2,Y_1,Y_2$.


In [2]:
E = ('A', 'B', 'C', 'D', 'X1', 'X2', 'Y1', 'Y2')
r = 4
ground = frozenset(E)
core = frozenset({'A', 'B', 'C', 'D'})
X = frozenset({'X1', 'X2'})
Y = frozenset({'Y1', 'Y2'})

zero_sets = frozenset(map(frozenset, (
    ('A', 'B', 'X1', 'X2'), ('C', 'D', 'X1', 'X2'),
    ('A', 'C', 'Y1', 'Y2'), ('B', 'D', 'Y1', 'Y2'))))
x_sets = frozenset(map(frozenset, (
    ('A', 'C', 'X1', 'X2'), ('A', 'D', 'X1', 'X2'),
    ('B', 'C', 'X1', 'X2'), ('B', 'D', 'X1', 'X2'))))
x2_sets = frozenset(map(frozenset, (
    ('A', 'B', 'Y1', 'Y2'), ('A', 'D', 'Y1', 'Y2'),
    ('B', 'C', 'Y1', 'Y2'), ('C', 'D', 'Y1', 'Y2'))))
six_pairs = frozenset(map(frozenset, (('A', 'B'), ('C', 'D'))))
five_pairs = frozenset(map(frozenset, (('A', 'C'), ('B', 'D'))))


def chi(S):
    S = frozenset(S)
    if len(S) != r or not S <= ground:
        raise ValueError('chi is defined on 4-subsets of E.')
    if S in zero_sets:
        return P_ZERO
    if S in x_sets:
        return x_power(1)
    if S in x2_sets:
        return x_power(2)
    mixed = len(S & X) == len(S & Y) == 1
    if mixed and S & core in six_pairs:
        return x_power(6)
    if mixed and S & core in five_pairs:
        return x_power(5)
    return P_ONE


In [3]:
coordinates = {frozenset(S): chi(S) for S in combinations(E, r)}
multiplicities = Counter(coordinates.values())
print('Coordinates:', len(coordinates))
print('Value | Multiplicity')
for value in (P_ZERO, P_ONE, *range(1, 7)):
    if multiplicities[value]:
        print(f'{p_label(value):5} | {multiplicities[value]}')
assert multiplicities == Counter({P_ZERO: 4, P_ONE: 42, 1: 4, 2: 4, 5: 8, 6: 8})


Coordinates: 70
Value | Multiplicity
0     | 4
1     | 42
x     | 4
x^2   | 4
x^5   | 8
x^6   | 8


## 3. What is checked

Baker–Lorscheid define a $P$-representation using **matroidal support**, alternation,
and the three-term Plücker relations; see [arXiv:2008.00014v1](https://arxiv.org/abs/2008.00014v1), p. 5.
These are the pasture-representation conditions, not a claim about all
higher-term relations.

The function `check_p_matroid(E, r, chi)` checks both of the following and
returns separate reports:

* The support $\mathcal B=\{B:\chi(B)\ne0\}$ is nonempty. For every
  $B_1,B_2\in\mathcal B$ and $e\in B_1\setminus B_2$, some
  $f\in B_2\setminus B_1$ satisfies $(B_1-\{e\})\cup\{f\}\in\mathcal B$.
* For every $(r-2)$-subset $J$ and every four-subset $\{a,b,c,d\}$ of
  $E\setminus J$, the three products
  $$\chi(Jab)\chi(Jcd),\qquad\chi(Jac)\chi(Jbd),\qquad\chi(Jad)\chi(Jbc)$$
  form a null triple, **including relations with zero products**.

Since $-1=1$, the subset function extends to an alternating tuple function by
assigning zero to repeated tuples and using the same value in every order of a
distinct tuple. Permutations of $a,b,c,d$ merely permute the three products.
Repeated-index relations reduce to $(0,0,0)$ or $(0,t,t)$, so are automatic.
Thus the enumeration uses
$$\binom{8}{2}\binom{6}{4}=420$$
three-term relations, without redundant permutations.

The individual functions `check_support_matroid` and `check_three_term_plucker`
are also available. The latter alone does **not** certify matroidal support.
All functions accept a callable on `frozenset` subsets or a complete mapping;
missing coordinates are errors, not implicit zeros. Read the Boolean verdict
in the returned dictionary, not the truth value of the dictionary itself.


In [4]:
report = check_p_matroid(E, r, chi)
support = report['support']
plucker = report['plucker']

print('Support size:', report['support_size'])
print('Support forms a matroid:', support['is_matroid'])
print('Basis-exchange checks:', support['basis_exchange_checks'])
print('Basis-exchange violations:', support['violations'])
print('Three-term relations checked:', plucker['relations_checked'])
print('Three-term violations:', plucker['violations'])
print('P-matroid:', report['is_p_matroid'])
print('\nNonzero products | Relations | Failures')
for k, item in plucker['by_nonzero_terms'].items():
    print(f"{k:16} | {item['checked']:9} | {item['failures']}")

assert report['is_p_matroid'], report
assert report['support_size'] == 66
assert support['basis_exchange_checks'] == 8712
assert support['violations'] == 0
assert plucker['relations_checked'] == 420
assert plucker['violations'] == 0
assert {k: v['checked'] for k, v in plucker['by_nonzero_terms'].items()} == {
    0: 0, 1: 0, 2: 142, 3: 278}


Support size: 66
Support forms a matroid: True
Basis-exchange checks: 8712
Basis-exchange violations: 0
Three-term relations checked: 420
Three-term violations: 0
P-matroid: True

Nonzero products | Relations | Failures
               0 |         0 | 0
               1 |         0 | 0
               2 |       142 | 0
               3 |       278 | 0


## Conclusion

The support of $\chi$ is a rank-$4$ matroid on $E$, and all three-term Plücker
relations hold over $P$. Hence $\chi$ is a $P$-representation in the sense of
Baker–Lorscheid. This is the end of the computational verification requested
here: no multiple-exchange property or higher-term relation is tested.

## Reference

Matthew Baker and Oliver Lorscheid, *Foundations of matroids, Part 1: Matroids
without large uniform minors*, [arXiv:2008.00014v1](https://arxiv.org/abs/2008.00014v1).
See p. 5 for $P$-representations and Definition 2.1 for pastures.
